In [6]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("Groq API key not found. Please check your .env file.")

# Paths
PDF_DOCS_PATH = "./documents"   # folder containing medical PDFs
CHROMA_PERSIST_PATH = "./db2/chroma_store"


In [4]:
# Load PDF docs
loader = DirectoryLoader(PDF_DOCS_PATH, glob="*.pdf", loader_cls=PyPDFLoader)
documents = loader.load()

# Split into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)

print(f" Loaded {len(documents)} documents, split into {len(docs)} chunks")


 Loaded 136 documents, split into 381 chunks


In [7]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# If DB already exists, load it. Otherwise, create it.
if os.path.exists(CHROMA_PERSIST_PATH):
    print(" Loading existing Chroma vector DB...")
    vectorstore = Chroma(persist_directory=CHROMA_PERSIST_PATH, embedding_function=embeddings)
else:
    print("Creating new Chroma DB...")
    vectorstore = Chroma.from_documents(docs, embeddings, persist_directory=CHROMA_PERSIST_PATH)
    vectorstore.persist()

print("Vector store ready!")



 Loading existing Chroma vector DB...


C:\Users\akmal\AppData\Local\Temp\ipykernel_37388\952181849.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=CHROMA_PERSIST_PATH, embedding_function=embeddings)


Vector store ready!


In [13]:
# Groq LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)

# Memory to maintain chat history
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Retrieval QA chain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    memory=memory
)

print(" Medical Chatbot is ready!")


 Medical Chatbot is ready!


In [ ]:
while True:
    query = input("You:what is fever ")
    if query.lower() in ["exit", "quit"]:
        print(" Goodbye!")
        break
    response = qa_chain.invoke({"question": query})
    print("Bot:", response["answer"])


In [12]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.3-70b-versatile"
)

print(llm.invoke("Hello, are you working?"))


content="Hello. Yes, I'm working and ready to help you with any questions or tasks you may have. How can I assist you today?" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 41, 'total_tokens': 70, 'completion_time': 0.059476245, 'prompt_time': 0.015623452, 'queue_time': 0.045036528, 'total_time': 0.075099697}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_2ddfbb0da0', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None} id='run--539033ac-595e-4069-a753-fb09ee23d0b1-0' usage_metadata={'input_tokens': 41, 'output_tokens': 29, 'total_tokens': 70}
